<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-09/AI_Modul_9.8_Praktikum_Hyperparameter_Tuning.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 9.8: Praktikum Regularisasi & Generalisasi Deep Learning
**Mata Kuliah:** AI Academy  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Memahami secara numerik disparitas antara galat empiris (*training loss*) dan galat generalisasi (*validation loss*) pada arsitektur jaringan saraf dalam.
2. Mengimplementasikan algoritma **$L_2$ Weight Decay**, **Inverted Dropout**, dan **Batch Normalization** dari prinsip dasar komputasi numerik NumPy.
3. Membangun modul **Early Stopping** untuk menangkap bobot model dengan generalisasi optimal sebelum terjadi *overfitting*.
4. Menganalisis pengaruh teknik regularisasi terhadap distribusi parameter bobot dan ketahanan prediksi pada data spektroskopi agrokompleks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Konfigurasi seed komputasi untuk reproduktibilitas hasil eksperimen
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Pustaka NumPy dan Matplotlib siap digunakan.")


## 1. Pembangkitan Dataset Agrokompleks: Spektrometri Vis-NIR Daun Kelapa Sawit
Kita membangkitkan dataset spektral reflektansi daun sawit sintetis yang merefleksikan 120 kanal panjang gelombang ($D = 120$) untuk $N = 400$ sampel tanaman. Target regresi adalah estimasi kadar Nitrogen daun (%).
Ukuran sampel yang terbatas dengan dimensi tinggi merepresentasikan kondisi ekstrem yang memicu *overfitting*.


In [ ]:
# Pembangkitan data spektral sintetis dengan multikolinearitas tinggi
N = 400
D = 120

# Basis spektral dasar
base_wave = np.linspace(400, 1000, D)
X = np.zeros((N, D))
for i in range(N):
    peak1 = np.exp(-((base_wave - 550)**2) / (2 * 40**2)) * np.random.uniform(0.6, 1.2)
    peak2 = np.exp(-((base_wave - 720)**2) / (2 * 60**2)) * np.random.uniform(0.4, 0.9)
    noise = np.random.normal(0, 0.03, D)
    X[i] = peak1 + peak2 + noise

# Fungsi target non-linier sejati berbasis kombinasi beberapa pita spektral
true_weights = np.zeros(D)
true_weights[20:30] = 1.8   # pita serapan hijau
true_weights[60:75] = -2.4  # red-edge klorofil
y = np.dot(X, true_weights) + 0.5 * (X[:, 25] * X[:, 70]) + np.random.normal(0, 0.15, N)
y = y.reshape(-1, 1)

# Pembagian data: Train (60%), Validation (20%), Test (20%)
perm = np.random.permutation(N)
X = X[perm]
y = y[perm]

n_train = int(0.6 * N)
n_val = int(0.2 * N)

X_train, y_train = X[:n_train], y[:n_train]
X_val, y_val = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
X_test, y_test = X[n_train+n_val:], y[n_train+n_val:]

# Standarisasi data fitur berbasis data latih
x_mean = np.mean(X_train, axis=0)
x_std = np.std(X_train, axis=0) + 1e-8

X_train = (X_train - x_mean) / x_std
X_val = (X_val - x_mean) / x_std
X_test = (X_test - x_mean) / x_std

print(f"Dimensi Data Latih   : {X_train.shape}, Target: {y_train.shape}")
print(f"Dimensi Data Validasi: {X_val.shape}, Target: {y_val.shape}")
print(f"Dimensi Data Uji     : {X_test.shape}, Target: {y_test.shape}")


## 2. Implementasi Modul Komputasi: Dropout, BatchNorm, dan Early Stopping
Kita membangun kelas-kelas komputasi numerik modular yang mendukung fase *train* dan *eval*.


In [ ]:
class InvertedDropout:
    """Lapisan Inverted Dropout dengan faktor penskalaan balik 1/(1-p)."""
    def __init__(self, p=0.0):
        self.p = p
        self.mask = None

    def forward(self, A, mode='train'):
        if mode == 'train' and self.p > 0.0:
            self.mask = (np.random.rand(*A.shape) >= self.p).astype(float)
            return (A * self.mask) / (1.0 - self.p)
        else:
            self.mask = None
            return A

    def backward(self, dA):
        if self.mask is not None:
            return (dA * self.mask) / (1.0 - self.p)
        return dA


class BatchNormalization:
    """Lapisan Batch Normalization dengan pelacakan Exponential Moving Average (EMA)."""
    def __init__(self, d, momentum=0.1, eps=1e-5):
        self.gamma = np.ones((1, d))
        self.beta = np.zeros((1, d))
        self.running_mean = np.zeros((1, d))
        self.running_var = np.ones((1, d))
        self.momentum = momentum
        self.eps = eps
        
        # Cache backward
        self.Z_norm = None
        self.var = None
        self.mu = None
        self.Z = None

    def forward(self, Z, mode='train'):
        if mode == 'train':
            self.Z = Z
            self.mu = np.mean(Z, axis=0, keepdims=True)
            self.var = np.var(Z, axis=0, keepdims=True)
            self.Z_norm = (Z - self.mu) / np.sqrt(self.var + self.eps)
            out = self.gamma * self.Z_norm + self.beta
            
            # Perbarui statistik berjalan
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * self.mu
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * self.var
            return out
        else:
            Z_norm = (Z - self.running_mean) / np.sqrt(self.running_var + self.eps)
            return self.gamma * Z_norm + self.beta

    def backward(self, dout):
        # Gradien parameter affine
        self.dgamma = np.sum(dout * self.Z_norm, axis=0, keepdims=True)
        self.dbeta = np.sum(dout, axis=0, keepdims=True)
        
        # Gradien terhadap input Z
        m = dout.shape[0]
        std_inv = 1.0 / np.sqrt(self.var + self.eps)
        dZ_norm = dout * self.gamma
        dvar = np.sum(dZ_norm * (self.Z - self.mu) * -0.5 * (std_inv**3), axis=0, keepdims=True)
        dmu = np.sum(dZ_norm * -std_inv, axis=0, keepdims=True) + dvar * np.mean(-2.0 * (self.Z - self.mu), axis=0, keepdims=True)
        dZ = dZ_norm * std_inv + dvar * 2.0 * (self.Z - self.mu) / m + dmu / m
        return dZ


class EarlyStopping:
    """Mekanisme pemantau keterpurukan validasi untuk mencegah overfitting."""
    def __init__(self, patience=15, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = np.inf
        self.early_stop = False
        self.best_weights = None

    def step(self, val_loss, model_weights):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_weights = [w.copy() for w in model_weights]
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True



## 3. Uji Verifikasi Invarian Ekspektasi Inverted Dropout
Kita buktikan secara empiris bahwa ekspektasi aktivasi lapisan saat pelatihan ekuivalen dengan saat evaluasi: $\mathbb{E}[\mathbf{\tilde{a}}] = \mathbf{a}$.


In [ ]:
sample_act = np.ones((1000, 10)) * 5.0
dropout_layer = InvertedDropout(p=0.4)

act_train = dropout_layer.forward(sample_act, mode='train')
act_eval = dropout_layer.forward(sample_act, mode='eval')

print(f"Rata-rata Ekspektasi Mode Pelatihan (p=0.4): {np.mean(act_train):.4f}")
print(f"Rata-rata Ekspektasi Mode Evaluasi         : {np.mean(act_eval):.4f}")
print(f"Selisih Absolut Relatif                    : {abs(np.mean(act_train) - np.mean(act_eval)):.6f}")


## 4. Konstruksi Arsitektur Deep Multi-Layer Perceptron (MLP)
Kita menyusun arsitektur jaringan saraf dalam [120, 64, 32, 1] dengan opsi konfigurasi regularisasi komposit.


In [ ]:
class DeepMLP:
    def __init__(self, layers=[120, 64, 32, 1], use_bn=False, dropout_rate=0.0, l2_lambda=0.0):
        self.layers = layers
        self.use_bn = use_bn
        self.dropout_rate = dropout_rate
        self.l2_lambda = l2_lambda
        
        self.W = []
        self.b = []
        self.bn_layers = []
        self.dropout_layers = []
        
        for l in range(len(layers) - 1):
            w = np.random.randn(layers[l], layers[l+1]) * np.sqrt(2.0 / layers[l])
            bias = np.zeros((1, layers[l+1]))
            self.W.append(w)
            self.b.append(bias)
            
            if l < len(layers) - 2:
                if self.use_bn:
                    self.bn_layers.append(BatchNormalization(layers[l+1]))
                if self.dropout_rate > 0.0:
                    self.dropout_layers.append(InvertedDropout(self.dropout_rate))

    def forward(self, X, mode='train'):
        self.A = [X]
        self.Z = []
        current = X
        
        for l in range(len(self.W) - 1):
            z = np.dot(current, self.W[l]) + self.b[l]
            self.Z.append(z)
            
            if self.use_bn:
                z = self.bn_layers[l].forward(z, mode=mode)
            
            a = np.maximum(0, z)
            
            if self.dropout_rate > 0.0:
                a = self.dropout_layers[l].forward(a, mode=mode)
                
            self.A.append(a)
            current = a
            
        z_out = np.dot(current, self.W[-1]) + self.b[-1]
        self.Z.append(z_out)
        self.A.append(z_out)
        return z_out

    def compute_loss(self, y_pred, y_true):
        m = y_true.shape[0]
        mse = np.mean((y_pred - y_true)**2)
        l2_cost = 0.0
        if self.l2_lambda > 0.0:
            for w in self.W:
                l2_cost += 0.5 * (self.l2_lambda / m) * np.sum(w**2)
        return mse + l2_cost

    def backward(self, y_pred, y_true, lr=0.01):
        m = y_true.shape[0]
        dA = 2.0 * (y_pred - y_true) / m
        
        dW_last = np.dot(self.A[-2].T, dA)
        db_last = np.sum(dA, axis=0, keepdims=True)
        if self.l2_lambda > 0.0:
            dW_last += (self.l2_lambda / m) * self.W[-1]
            
        dA_prev = np.dot(dA, self.W[-1].T)
        
        self.W[-1] -= lr * dW_last
        self.b[-1] -= lr * db_last
        
        for l in reversed(range(len(self.W) - 1)):
            if self.dropout_rate > 0.0:
                dA_prev = self.dropout_layers[l].backward(dA_prev)
                
            dZ = dA_prev * (self.Z[l] > 0)
            
            if self.use_bn:
                dZ = self.bn_layers[l].backward(dZ)
                self.bn_layers[l].gamma -= lr * self.bn_layers[l].dgamma
                self.bn_layers[l].beta -= lr * self.bn_layers[l].dbeta
                
            dW = np.dot(self.A[l].T, dZ)
            db = np.sum(dZ, axis=0, keepdims=True)
            if self.l2_lambda > 0.0:
                dW += (self.l2_lambda / m) * self.W[l]
                
            dA_prev = np.dot(dZ, self.W[l].T)
            
            self.W[l] -= lr * dW
            self.b[l] -= lr * db


## 5. Eksperimen Komparasi: Model Tanpa Regularisasi vs Model Teregulasi
Kita melatih dua model secara paralel selama 350 epoch:
1. **Model A (Baseline):** Tanpa regularisasi ($L_2 = 0$, Dropout = 0, Tanpa BN).
2. **Model B (Regularized):** $L_2 = 0.05$, Dropout = $0.2$, Menggunakan BatchNorm, dan Early Stopping.


In [ ]:
epochs = 350
learning_rate = 0.02

model_unreg = DeepMLP(layers=[120, 64, 32, 1], use_bn=False, dropout_rate=0.0, l2_lambda=0.0)
model_reg = DeepMLP(layers=[120, 64, 32, 1], use_bn=True, dropout_rate=0.2, l2_lambda=0.05)
early_stopper = EarlyStopping(patience=30, min_delta=1e-3)

history_unreg = {'train_loss': [], 'val_loss': []}
history_reg = {'train_loss': [], 'val_loss': []}
stopped_epoch = epochs

for epoch in range(epochs):
    # Model Tanpa Regularisasi
    pred_train_u = model_unreg.forward(X_train, mode='train')
    loss_train_u = model_unreg.compute_loss(pred_train_u, y_train)
    model_unreg.backward(pred_train_u, y_train, lr=learning_rate)
    
    pred_val_u = model_unreg.forward(X_val, mode='eval')
    loss_val_u = np.mean((pred_val_u - y_val)**2)
    history_unreg['train_loss'].append(loss_train_u)
    history_unreg['val_loss'].append(loss_val_u)
    
    # Model Teregulasi
    if not early_stopper.early_stop:
        pred_train_r = model_reg.forward(X_train, mode='train')
        loss_train_r = model_reg.compute_loss(pred_train_r, y_train)
        model_reg.backward(pred_train_r, y_train, lr=learning_rate)
        
        pred_val_r = model_reg.forward(X_val, mode='eval')
        loss_val_r = np.mean((pred_val_r - y_val)**2)
        history_reg['train_loss'].append(loss_train_r)
        history_reg['val_loss'].append(loss_val_r)
        
        early_stopper.step(loss_val_r, model_reg.W)
        stopped_epoch = epoch

if early_stopper.best_weights is not None:
    model_reg.W = early_stopper.best_weights

print("Pelatihan selesai secara optimal!")
print(f"Early Stopping terpicu pada epoch ke-{stopped_epoch} dengan Best Val MSE: {early_stopper.best_loss:.4f}")


## 6. Visualisasi Kurva Generalisasi dan Efek Penalti Bobot
Kita menyandingkan kurva evaluasi antara model tanpa regularisasi dan model teregulasi, serta mengamati penyusutan magnitudo bobot (*weight shrinkage*).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Kurva Evaluasi Train vs Val
axes[0].plot(history_unreg['train_loss'], 'r--', alpha=0.7, label='Unreg - Train MSE')
axes[0].plot(history_unreg['val_loss'], 'r-', label='Unreg - Val MSE (Overfit)')
axes[0].plot(history_reg['train_loss'], 'g--', alpha=0.7, label='Reg - Train Loss')
axes[0].plot(history_reg['val_loss'], 'g-', label='Reg - Val MSE (Generalize)')
axes[0].axvline(stopped_epoch, color='navy', linestyle=':', label=f'Early Stop (Epoch {stopped_epoch})')
axes[0].set_title('Dinamika Kurva Belajar & Pencegahan Overfitting', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch Iterasi')
axes[0].set_ylabel('Mean Squared Error')
axes[0].set_ylim(0, 5.0)
axes[0].legend()

# Plot 2: Histogram Distribusi Bobot Parameter
w_unreg_flat = np.concatenate([w.flatten() for w in model_unreg.W])
w_reg_flat = np.concatenate([w.flatten() for w in model_reg.W])

axes[1].hist(w_unreg_flat, bins=50, color='red', alpha=0.5, density=True, label=f'Tanpa Regularisasi (Std: {np.std(w_unreg_flat):.2f})')
axes[1].hist(w_reg_flat, bins=50, color='green', alpha=0.6, density=True, label=f'Dengan L2 Decay (Std: {np.std(w_reg_flat):.2f})')
axes[1].set_title('Efek L2 Weight Decay terhadap Dispersi Bobot', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Nilai Bobot Parameter')
axes[1].set_ylabel('Densitas Frekuensi')
axes[1].legend()

plt.tight_layout()
plt.show()


## 7. Evaluasi Kinerja Generalisasi pada Data Uji Independen
Kita menguji kedua model pada himpunan data uji (*test set*) yang belum pernah dilihat sama sekali.


In [ ]:
pred_test_unreg = model_unreg.forward(X_test, mode='eval')
pred_test_reg = model_reg.forward(X_test, mode='eval')

mse_test_u = np.mean((pred_test_unreg - y_test)**2)
r2_test_u = 1 - (np.sum((y_test - pred_test_unreg)**2) / np.sum((y_test - np.mean(y_test))**2))

mse_test_r = np.mean((pred_test_reg - y_test)**2)
r2_test_r = 1 - (np.sum((y_test - pred_test_reg)**2) / np.sum((y_test - np.mean(y_test))**2))

print("=================================================================")
print("             HASIL EVALUASI PADA DATA UJI INDEPENDEN            ")
print("=================================================================")
print(f"Model Tanpa Regularisasi (Baseline) -> Test MSE: {mse_test_u:.4f} | R2: {r2_test_u:.4f}")
print(f"Model Teregulasi (L2+Drop+BN+ES)    -> Test MSE: {mse_test_r:.4f} | R2: {r2_test_r:.4f}")
print("=================================================================")

peningkatan_r2 = (r2_test_r - r2_test_u) / abs(r2_test_u) * 100 if r2_test_u != 0 else 0
print(f"Peningkatan Kualitas Generalisasi: {peningkatan_r2:.1f}%")
